# 03 数据分析 & 可视化

**目标**：
1. 对原始结果评分（EM / Contains）
2. 生成 NIAH 热力图（signature 图表）
3. 分析跨模型准确率 vs 上下文长度
4. 验证 "Lost in the Middle" 位置偏差
5. 导出所有图表到 `results/figures/`

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio

from src.metrics import score_results, print_summary, aggregate_by_position_bucket
from src.visualize import (
    plot_niah_heatmap,
    plot_niah_heatmap_interactive,
    plot_accuracy_by_length,
    plot_position_bias,
)

# Plotly 在 notebook 中内联显示
pio.renderers.default = 'notebook'

print('模块加载完毕')

## Step 1：加载原始结果 & 评分

In [ ]:
raw_path = '../results/raw/raw_results.csv'

if not Path(raw_path).exists():
    raise FileNotFoundError('请先运行 02_eval_runner.ipynb 获取原始结果')

df_raw = pd.read_csv(raw_path)
print(f'原始结果: {len(df_raw)} 条, 模型: {df_raw["model"].unique().tolist()}')

# 打分
df = score_results(df_raw)

# 保存评分结果
out_dir = Path('../results/processed')
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / 'scored_results.csv', index=False, encoding='utf-8-sig')
print(f'✅ 评分结果已保存')

# 整体摘要
print_summary(df)

## Step 2：失败样本分析

In [ ]:
# 查看 contains_score=0 的失败样本
failures = df[df['contains_score'] == 0]
print(f'失败样本: {len(failures)} / {len(df)} ({len(failures)/len(df)*100:.1f}%)')

if len(failures) > 0:
    print('\n失败样本示例（前 5 条）：')
    cols = ['model', 'context_length', 'depth_pct', 'expected_answer', 'model_response']
    display(failures[cols].head(5))

## Step 3：NIAH 热力图

这是整个项目的 **核心图表**，清晰展示模型在不同上下文长度 × needle 位置下的准确率。

In [ ]:
# 静态热力图（用于 README / 论文）
for model in df['model'].unique():
    fig = plot_niah_heatmap(df, model, save=True)
    plt.show()

In [ ]:
# 交互式热力图（Plotly，可以 hover 查看详情）
for model in df['model'].unique():
    fig = plot_niah_heatmap_interactive(df, model, save_html=True)
    fig.show()

## Step 4：准确率 vs 上下文长度（跨模型对比）

In [ ]:
fig = plot_accuracy_by_length(df, save=True)
plt.show()

## Step 5："Lost in the Middle" 位置偏差分析

In [ ]:
# 柱状图
fig = plot_position_bias(df, save=True)
plt.show()

In [ ]:
# 数值表格
pos_df = aggregate_by_position_bucket(df)
pivot = pos_df.pivot(index='model', columns='position_bucket', values='contains_score') * 100
pivot.columns.name = None
pivot = pivot.round(1)

print('各模型在不同位置的准确率 (%)')
display(pivot)

# 位置偏差：中间 vs 平均
print('\n"Lost in the Middle" 偏差（中间准确率 - 全局平均）：')
for model in df['model'].unique():
    sub = df[df['model'] == model]
    overall = sub['contains_score'].mean() * 100
    middle = sub[(sub['depth_pct'] > 20) & (sub['depth_pct'] <= 70)]['contains_score'].mean() * 100
    drop = middle - overall
    print(f'  {model:12s}: 全局 {overall:.1f}%  中间 {middle:.1f}%  差值 {drop:+.1f}%')

## Step 6：准确率随深度变化趋势（折线图）

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for model, sub in df.groupby('model'):
    depth_acc = sub.groupby('depth_pct')['contains_score'].mean() * 100
    depth_acc = depth_acc.sort_index()
    ax.plot(depth_acc.index, depth_acc.values, marker='o', label=model, linewidth=2)

ax.set_xlabel('Needle 插入深度 (%)', fontsize=12)
ax.set_ylabel('准确率 (%)', fontsize=12)
ax.set_title('准确率 vs Needle 深度（"Lost in the Middle" 可视化）', fontsize=13)
ax.legend(fontsize=11)
ax.axvspan(20, 70, alpha=0.08, color='red', label='中间区域')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 108)

from pathlib import Path
Path('../results/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('../results/figures/depth_accuracy_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 已保存: results/figures/depth_accuracy_curve.png')

## ✅ 分析完成

所有图表已保存至 `results/figures/`：
- `niah_heatmap_<model>.png` — NIAH 热力图（主图）
- `niah_heatmap_<model>_interactive.html` — 交互式热力图
- `accuracy_by_length.png` — 准确率 vs 上下文长度
- `position_bias.png` — 位置偏差柱状图
- `depth_accuracy_curve.png` — 深度-准确率曲线

下一步：打开 `04_report.ipynb` 撰写分析报告